# Faz 6 — Harici Test (External Holdout Evaluation)

Tüm eğitilmiş modelleri harici test seti üzerinde değerlendirir ve
sonuçları `outputs/results/` altına JSON olarak kaydeder.

**Faz8_Makale_Analiz.ipynb bu JSON dosyalarını kullanır.**

### Beklenen sonuç dosyaları
- `outputs/results/yolo_results.json`
- `outputs/results/swintransformer_results.json`
- `outputs/results/nnunet_results.json`
- `outputs/results/mednext_results.json`
- `outputs/results/ablation_results.json`

In [ ]:
import os, sys, json
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/abdomen')
    DATA_DIR     = Path('/content/drive/MyDrive/abdomenDataSet')
elif IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/abdomen')
    DATA_DIR     = Path('/kaggle/input/abdomen-dataset')
else:  # LOCAL
    import platform
    if platform.system() == 'Windows':
        PROJECT_ROOT = Path(r'D:/makale-pdf/Proje/abdomen')
        DATA_DIR     = Path(r'D:/makale-pdf/Proje/abdomenDataSet')
    else:  # macOS / Linux
        PROJECT_ROOT = Path('/Users/ramazanpolat/Desktop/datasets/abdomen')
        DATA_DIR     = Path('/Users/ramazanpolat/Desktop/datasets/abdomenDataSet')

WORK_DIR    = PROJECT_ROOT if IS_LOCAL else Path('/content' if IS_COLAB else '/kaggle/working')
OUT_DIR     = WORK_DIR / 'outputs'
SPLIT_DIR   = OUT_DIR / 'splits'
CKPT_DIR    = OUT_DIR / 'checkpoints'
RESULTS_DIR = OUT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault('ABDOMEN_PROJECT_ROOT', str(PROJECT_ROOT))
os.environ.setdefault('ABDOMEN_DATA_ROOT',    str(WORK_DIR))
os.environ.setdefault('ABDOMEN_SPLIT_DIR',    str(SPLIT_DIR))

FOLD = 0
SUPER_CLASSES = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

print(f'Ortam: {"LOCAL" if IS_LOCAL else "COLAB" if IS_COLAB else "KAGGLE"}')
print(f'RESULTS_DIR: {RESULTS_DIR}')

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.config import SUPER_CLASSES as CFG_CLASSES
from src.evaluation import Evaluator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Test split CSV yükle
test_csv = SPLIT_DIR / f'fold{FOLD}_test.csv'
if not test_csv.exists():
    # Harici holdout test set CSV
    test_csv = SPLIT_DIR / 'holdout_test.csv'
if not test_csv.exists():
    print(f'[UYARI] Test CSV bulunamadı: {test_csv}')
    print('splits.csv içindeki split==test satırları kullanılıyor...')
    splits_csv = SPLIT_DIR / 'splits.csv'
    if splits_csv.exists():
        df_all = pd.read_csv(splits_csv)
        df_test = df_all[df_all['split'] == 'test']
        print(f'Test seti: {len(df_test)} case')
    else:
        df_test = None
        print('[HATA] splits.csv de bulunamadı!')
else:
    df_test = pd.read_csv(test_csv)
    print(f'Test seti: {len(df_test)} satır — {test_csv.name}')

## 1. SwinTransformer Değerlendirmesi

In [ ]:
import time, warnings
from src.train_cls import build_model, ClsConfig
from src.datasets import AbdomenClsDataset
import torchvision.transforms as T

def measure_efficiency(model, input_size=224, n_warmup=10, n_runs=50):
    """Parametre sayısı, çıkarım süresi (ms/img), peak VRAM ölçer."""
    params_M = sum(p.numel() for p in model.parameters()) / 1e6

    dummy = torch.zeros(1, 3, input_size, input_size, device=DEVICE)
    model.eval()
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)

    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    ms_per_img = (t1 - t0) / n_runs * 1000
    peak_vram  = (torch.cuda.max_memory_allocated() / 1e9) if DEVICE == 'cuda' else 0.0
    return round(params_M, 2), round(ms_per_img, 2), round(peak_vram, 3)


def run_cls_eval(model_name, backbone, ckpt_path, df_test, fold=0):
    """Sınıflandırma modelini test setinde değerlendirir.

    Kaydedilen JSON anahtarları:
      top5_mean_f1, per_class_f1, thresholds_used,
      predictions.probs / predictions.labels  (PR & kalibrasyon için),
      efficiency.params_M / .inference_ms / .peak_vram_GB
    """
    if df_test is None:
        print(f'[ATLA] {model_name}: test verisi yok'); return None
    if not Path(ckpt_path).exists():
        print(f'[ATLA] {model_name}: checkpoint bulunamadı: {ckpt_path}'); return None

    cfg   = ClsConfig(backbone=backbone, fold=fold)
    model = build_model(cfg).to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt.get('model_state_dict', ckpt))
    model.eval()

    # Verimlilik ölç
    params_M, infer_ms, peak_vram = measure_efficiency(model)
    print(f'  Params: {params_M:.1f}M | Inference: {infer_ms:.1f} ms | VRAM: {peak_vram:.2f} GB')

    # Threshold yükle
    thresh_path = Path(ckpt_path).parent / f'{backbone}_thresholds_fold{fold}.json'
    if thresh_path.exists():
        with open(thresh_path) as f:
            thresholds = json.load(f)
        print(f'  Thresholds yüklendi: {thresh_path.name}')
    else:
        thresholds = {c: 0.5 for c in SUPER_CLASSES}
        print('  [UYARI] Threshold dosyası yok, 0.5 kullanılıyor')

    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    dataset = AbdomenClsDataset(df_test, transform=transform)
    loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=model_name):
            imgs   = batch['image'].to(DEVICE)
            labels = batch['labels']
            probs  = torch.sigmoid(model(imgs)).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labels.numpy())

    preds  = np.concatenate(all_preds,  axis=0)
    labels = np.concatenate(all_labels, axis=0)

    per_class_f1 = {}
    for i, cls in enumerate(SUPER_CLASSES):
        tau      = thresholds.get(cls, 0.5)
        pred_bin = (preds[:, i] >= tau).astype(int)
        tp = int(((pred_bin == 1) & (labels[:, i] == 1)).sum())
        fp = int(((pred_bin == 1) & (labels[:, i] == 0)).sum())
        fn = int(((pred_bin == 0) & (labels[:, i] == 1)).sum())
        prec = tp / max(tp + fp, 1)
        rec  = tp / max(tp + fn, 1)
        f1   = 2 * prec * rec / max(prec + rec, 1e-9)
        per_class_f1[cls] = round(f1, 4)

    top5_mean_f1 = round(float(np.mean(list(per_class_f1.values()))), 4)

    result = {
        'model': model_name,
        'backbone': backbone,
        'top5_mean_f1': top5_mean_f1,
        'per_class_f1': per_class_f1,
        'thresholds_used': thresholds,
        # ham olasılıklar — PR eğrisi ve kalibrasyon için
        'predictions': {
            'probs':  preds.tolist(),
            'labels': labels.tolist(),
        },
        # verimlilik metrikleri
        'efficiency': {
            'params_M':      params_M,
            'inference_ms':  infer_ms,
            'peak_vram_GB':  peak_vram,
        },
    }
    print(f'  Top-5 Mean F1: {top5_mean_f1:.4f}')
    return result


# SwinTransformer değerlendirmesi
swin_ckpt = CKPT_DIR / f'swin_base_fold{FOLD}_best.pth'
swin_result = run_cls_eval('SwinTransformer', 'swin_base', swin_ckpt, df_test, FOLD)
if swin_result:
    with open(RESULTS_DIR / 'swintransformer_results.json', 'w') as f:
        json.dump(swin_result, f, indent=2)
    print('  Kaydedildi: swintransformer_results.json')

## 2. YOLO Değerlendirmesi

In [ ]:
def run_yolo_eval(ckpt_path, det_data_dir, df_test):
    """YOLO modelini test setinde değerlendirir."""
    try:
        from ultralytics import YOLO
    except ImportError:
        print('[HATA] ultralytics yüklü değil: pip install ultralytics')
        return None

    if not Path(ckpt_path).exists():
        print(f'[ATLA] YOLO checkpoint bulunamadı: {ckpt_path}')
        return None

    model = YOLO(str(ckpt_path))

    # Test görüntülerini bul
    test_img_dir = Path(det_data_dir) / 'test' / 'images'
    if not test_img_dir.exists():
        print(f'[ATLA] YOLO test görüntü dizini bulunamadı: {test_img_dir}')
        return None

    results = model.val(data=str(Path(det_data_dir) / 'data.yaml'),
                       split='test', save_json=True, verbose=False)

    # metrics'ten F1 çıkar
    try:
        maps = results.box.maps  # per-class mAP@0.5
        per_class_f1 = {cls: round(float(maps[i]), 4) for i, cls in enumerate(SUPER_CLASSES)
                        if i < len(maps)}
        top5_mean_f1 = round(float(np.mean(list(per_class_f1.values()))), 4)
    except Exception as e:
        print(f'[UYARI] YOLO metrics parse hatası: {e}')
        per_class_f1 = {cls: 0.0 for cls in SUPER_CLASSES}
        top5_mean_f1 = 0.0

    yolo_result = {
        'model': 'YOLO',
        'top5_mean_f1': top5_mean_f1,
        'per_class_f1': per_class_f1,
    }
    print(f'YOLO Top-5 Mean F1: {top5_mean_f1:.4f}')
    return yolo_result

yolo_ckpt   = CKPT_DIR / 'yolo' / 'train' / 'weights' / 'best.pt'
det_data    = OUT_DIR / 'det_data'
yolo_result = run_yolo_eval(yolo_ckpt, det_data, df_test)
if yolo_result:
    with open(RESULTS_DIR / 'yolo_results.json', 'w') as f:
        json.dump(yolo_result, f, indent=2)
    print('Kaydedildi: yolo_results.json')

## 3. nnUNet Değerlendirmesi

In [ ]:
def run_nnunet_eval(results_dir_nnunet, df_test):
    """nnUNet tahmin sonuçlarını yükler ve F1 hesaplar."""
    pred_dir = Path(results_dir_nnunet)
    if not pred_dir.exists():
        print(f'[ATLA] nnUNet tahmin dizini bulunamadı: {pred_dir}')
        return None

    # nnUNet tahmin NIfTI'larından BB çıkar ve F1 hesapla
    try:
        from src.nnunet import extract_detections_from_segmentation
        pred_df, gt_df = extract_detections_from_segmentation(pred_dir, df_test)
        from src.evaluation import Evaluator
        ev = Evaluator(gt_df)
        top5_f1, per_cls = ev.top5_mean_f1(pred_df)
        nnunet_result = {
            'model': 'nnUNet',
            'top5_mean_f1': round(float(top5_f1), 4),
            'per_class_f1': {c: round(float(v), 4) for c, v in per_cls.items()},
        }
        print(f'nnUNet Top-5 Mean F1: {top5_f1:.4f}')
        return nnunet_result
    except Exception as e:
        print(f'[HATA] nnUNet değerlendirme: {e}')
        return None

nnunet_pred_dir = OUT_DIR / 'nnunet_predictions' / 'test'
nnunet_result = run_nnunet_eval(nnunet_pred_dir, df_test)
if nnunet_result:
    with open(RESULTS_DIR / 'nnunet_results.json', 'w') as f:
        json.dump(nnunet_result, f, indent=2)
    print('Kaydedildi: nnunet_results.json')

## 4. MedNeXt Değerlendirmesi

In [ ]:
def run_mednext_eval(pred_dir, df_test):
    """MedNeXt tahmin sonuçlarını yükler ve F1 hesaplar."""
    pred_dir = Path(pred_dir)
    if not pred_dir.exists():
        print(f'[ATLA] MedNeXt tahmin dizini bulunamadı: {pred_dir}')
        return None

    try:
        from src.nnunet import extract_detections_from_segmentation
        pred_df, gt_df = extract_detections_from_segmentation(pred_dir, df_test)
        from src.evaluation import Evaluator
        ev = Evaluator(gt_df)
        top5_f1, per_cls = ev.top5_mean_f1(pred_df)
        result = {
            'model': 'MedNeXt',
            'top5_mean_f1': round(float(top5_f1), 4),
            'per_class_f1': {c: round(float(v), 4) for c, v in per_cls.items()},
        }
        print(f'MedNeXt Top-5 Mean F1: {top5_f1:.4f}')
        return result
    except Exception as e:
        print(f'[HATA] MedNeXt değerlendirme: {e}')
        return None

mednext_pred_dir = OUT_DIR / 'mednext_predictions' / 'test'
mednext_result = run_mednext_eval(mednext_pred_dir, df_test)
if mednext_result:
    with open(RESULTS_DIR / 'mednext_results.json', 'w') as f:
        json.dump(mednext_result, f, indent=2)
    print('Kaydedildi: mednext_results.json')

## 5. Ablasyon Değerlendirmesi (Makale 2)

4 farklı SwinTransformer konfigürasyonu test seti üzerinde değerlendirilir.

In [ ]:
ABLATION_CONFIGS = [
    {
        'name': 'Baseline (BCE)',
        'backbone': 'swin_base',
        'ckpt': CKPT_DIR / f'ablation_baseline_fold{FOLD}_best.pth',
        'thresh_json': None,
    },
    {
        'name': 'FocalBCE',
        'backbone': 'swin_base',
        'ckpt': CKPT_DIR / f'ablation_focalbce_fold{FOLD}_best.pth',
        'thresh_json': None,
    },
    {
        'name': 'FocalBCE + WeightedSampler',
        'backbone': 'swin_base',
        'ckpt': CKPT_DIR / f'ablation_focalbce_sampler_fold{FOLD}_best.pth',
        'thresh_json': None,
    },
    {
        'name': 'FocalBCE + Sampler + ThreshTuning',
        'backbone': 'swin_base',
        'ckpt': CKPT_DIR / f'ablation_focalbce_sampler_fold{FOLD}_best.pth',
        'thresh_json': CKPT_DIR / f'ablation_focalbce_sampler_thresholds_fold{FOLD}.json',
    },
]

def eval_cls_model_simple(backbone, ckpt_path, thresh_json, df_test, config_name):
    """Basit değerlendirme — run_cls_eval'i tekrar kullanır."""
    if df_test is None:
        return None
    if not Path(ckpt_path).exists():
        print(f'  [ATLA] {config_name}: {Path(ckpt_path).name} bulunamadı')
        return None

    from src.train_cls import build_model, ClsConfig
    from src.datasets import AbdomenClsDataset
    import torchvision.transforms as T

    cfg   = ClsConfig(backbone=backbone, fold=FOLD)
    model = build_model(cfg).to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt.get('model_state_dict', ckpt))
    model.eval()

    if thresh_json and Path(thresh_json).exists():
        with open(thresh_json) as f:
            thresholds = json.load(f)
    else:
        thresholds = {c: 0.5 for c in SUPER_CLASSES}

    transform = T.Compose([
        T.Resize((224,224)), T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    dataset = AbdomenClsDataset(df_test, transform=transform)
    loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=config_name, leave=False):
            imgs   = batch['image'].to(DEVICE)
            labels = batch['labels']
            probs  = torch.sigmoid(model(imgs)).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labels.numpy())

    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    per_class_f1 = {}
    for i, cls in enumerate(SUPER_CLASSES):
        tau     = thresholds.get(cls, 0.5)
        pred_b  = (preds[:, i] >= tau).astype(int)
        tp = int(((pred_b==1)&(labels[:,i]==1)).sum())
        fp = int(((pred_b==1)&(labels[:,i]==0)).sum())
        fn = int(((pred_b==0)&(labels[:,i]==1)).sum())
        prec = tp / max(tp+fp, 1)
        rec  = tp / max(tp+fn, 1)
        f1   = 2*prec*rec / max(prec+rec, 1e-9)
        per_class_f1[cls] = round(f1, 4)

    top5 = round(float(np.mean(list(per_class_f1.values()))), 4)
    print(f'  {config_name}: Top-5 Mean F1 = {top5:.4f}')
    return {'name': config_name, 'top5_mean_f1': top5, 'per_class_f1': per_class_f1}

ablation_results = []
for cfg in ABLATION_CONFIGS:
    r = eval_cls_model_simple(
        backbone=cfg['backbone'],
        ckpt_path=cfg['ckpt'],
        thresh_json=cfg.get('thresh_json'),
        df_test=df_test,
        config_name=cfg['name']
    )
    if r:
        ablation_results.append(r)

if ablation_results:
    abl_out = {'configs': ablation_results}
    with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
        json.dump(abl_out, f, indent=2)
    print(f'\nAblasyon sonuçları kaydedildi: ablation_results.json ({len(ablation_results)} konfigürasyon)')
else:
    print('[UYARI] Hiçbir ablasyon checkpoint bulunamadı.')

In [ ]:
print('=== Faz 6 Özeti ===')
for fname in sorted(RESULTS_DIR.glob('*.json')):
    with open(fname) as f:
        d = json.load(f)
    model = d.get('model', fname.stem)
    f1    = d.get('top5_mean_f1', '—')
    print(f'  {model:30s}: Top-5 Mean F1 = {f1}')

print(f'\nSonuç dosyaları: {RESULTS_DIR}')
print('Şimdi Faz8_Makale_Analiz.ipynb çalıştırabilirsiniz.')